# Building a Transformer from Scratch

A step-by-step journey from the simplest possible language model (bigram) to a GPT-style transformer, adding one concept at a time. Each step introduces a single idea, trains a model, and generates text so you can see the impact.

| Step | Model | Key Addition |
|------|-------|-------------|
| 0 | Bigram | Embedding lookup — no context |
| 1 | Uniform Attention | Averaging previous tokens via matrix multiply |
| 2 | Self-Attention | Learned queries, keys, and values |
| 3 | Multi-Head Attention | Multiple attention heads in parallel |
| 4 | + Feed-Forward | Per-token MLP after attention |
| 5 | + Residual + LayerNorm | Skip connections for stable training |
| 6 | **Full GPT** | Stacked layers, dropout, scaled up |

We only need PyTorch.

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

## Data

We load `input.txt` — the [tiny Shakespeare](https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt) dataset (~1.1 MB of Shakespeare plays). This is our entire training corpus.

In [3]:
text = open('input.txt', 'r').read()
print(f'{len(text)} characters')
print(text[:200])

1115394 characters
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


### Tokenizer

Every unique character gets an integer ID. `encode` converts text to numbers, `decode` converts back. With 65 unique characters, our vocabulary is tiny — real models use subword tokenizers with 50k+ tokens.

In [4]:
chars = sorted(set(text))
vocab_size = len(chars)
print(f'Vocab size: {vocab_size}')
print(''.join(chars))

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

Vocab size: 65

 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz


### Train/Val Split

We encode the entire text into a tensor of integers, then split 90/10 into training and validation sets. No shuffling — we just take the first 90% for training.

In [5]:
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]
print(f'Train: {len(train_data)} tokens, Val: {len(val_data)} tokens')

Train: 1003854 tokens, Val: 111540 tokens


### Batching

`get_batch` grabs random chunks of text and creates input/target pairs. The target is just the input shifted by one — at each position, the model should predict the next character.

In [6]:
block_size = 8
batch_size = 32

def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i+block_size] for i in ix])
    y = torch.stack([d[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print(f'x shape: {xb.shape}, y shape: {yb.shape}')

x shape: torch.Size([32, 8]), y shape: torch.Size([32, 8])


## Step 0: Bigram Model

**What changes:** This is the baseline — no attention, no context.

**How it works:** The model is a single lookup table (`nn.Embedding`) of shape `(65, 65)` — one row per character in our vocabulary. When the model sees character `e` (index 48), it returns row 48 of the table: 65 scores, one per possible next character. The highest-scoring character is the most likely to follow `e`. Crucially, the model only looks at the current character — it has no idea what came before it. The word `"the"` and `"she"` would produce the exact same prediction after `e`.

**How training works:** We use a standard PyTorch loop. Each step: grab a random batch of text chunks, run them through the model to get predictions, measure how wrong those predictions are (cross-entropy loss), compute gradients via backpropagation, and nudge the weights to make better predictions next time. The optimizer (AdamW) handles the weight updates. Over 10,000 steps, the model learns character-pair statistics: `t` is often followed by `h`, `q` by `u`, etc.

**How generation works:** Start with a seed character. Get the model's prediction (65 scores), convert to probabilities with softmax, randomly sample one character from that distribution (so generation isn't deterministic), append it, and repeat.

In [ ]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # A 65x65 lookup table: row i = scores for "what comes after character i"
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        # idx is (B, T) — a batch of B sequences, each T characters long
        # Look up each character in the table → (B, T, 65) scores
        logits = self.token_embedding_table(idx)

        if targets is None:
            loss = None
        else:
            # PyTorch's cross_entropy wants flat inputs: (B*T, 65) and (B*T,)
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B*T, C), targets.view(B*T))

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _ = self(idx)
            logits = logits[:, -1, :]           # only the last position's prediction
            probs = F.softmax(logits, dim=-1)    # convert scores → probabilities
            idx_next = torch.multinomial(probs, num_samples=1)  # sample one token
            idx = torch.cat((idx, idx_next), dim=1)             # append to sequence
        return idx

model = BigramLanguageModel(vocab_size)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

### Generate before training (random baseline)

In [8]:
idx = torch.zeros((1, 1), dtype=torch.long)
print(decode(model.generate(idx, max_new_tokens=100)[0].tolist()))


Q&bQ.NPDTabBh-VhJMq?Mq!K'C$mQYKbvuVPD?g DeaWvapZqXK
INpmVsgP
r
R
VR?jw3JawiD$mD; nE&Ljx3KNxHGZFgPUJC


### Training

Standard training loop: sample a batch, compute loss, backpropagate, update weights. 10,000 steps with AdamW optimizer.

In [ ]:
# AdamW: Adam optimizer with weight decay — the standard for training language models
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

for step in range(10000):
    xb, yb = get_batch('train')       # random batch of (input, target) pairs
    logits, loss = model(xb, yb)       # forward pass: predictions + how wrong they are
    optimizer.zero_grad(set_to_none=True)  # clear gradients from previous step
    loss.backward()                    # backprop: compute how each weight contributed to the error
    optimizer.step()                   # nudge weights to reduce the error

    if step % 2000 == 0 or step == 9999:
        print(f'Step {step:5d}  loss={loss.item():.4f}')

### Generate after training

The model has learned which characters tend to follow which. You'll see common letter pairs and spaces in roughly the right places, but no real words — a bigram has no memory beyond one character.

In [10]:
idx = torch.zeros((1, 1), dtype=torch.long)
print(decode(model.generate(idx, max_new_tokens=300)[0].tolist()))


Ulde.
CI o hishis?

INI, f
NINIn, vepeendicho SHAroz?
Shy, foraunga M$VOUSThas, ce th:
Abe n:

IFoy
Poworee s brenk
llofour wame b&$D chatreay, e his me avqur mellvey,

Od, t'bereso Sth ageyst bot m

TES:
HUSLY; d
I t ST ind.
ANELAn hanck re one hoofeld,
POren.
MESIs pes yof wan.

SA:
Oforyse
D LO:



## Step 1: Uniform Attention

**What changes:** Tokens can now see previous tokens — but with fixed, equal weights.

**How it works:** The bigram model's problem is that each token is an island — position 5 has no idea what's at positions 0–4. We fix this by letting each token **average** all the tokens that came before it.

We build a weight matrix `wei` of shape `(T, T)` where row `t` says "how much should position `t` look at each other position?". We fill the upper triangle (future positions) with `-inf`, then apply softmax so each row sums to 1. Since all non-masked values start as zero (equal), softmax gives uniform weights: position 3 gets `[0.25, 0.25, 0.25, 0.25, 0, 0, 0, 0]` — an equal average of positions 0–3.

Multiplying `wei @ x` replaces each token's embedding with the average of all previous embeddings. This is the core attention mechanic — a weight matrix times the embeddings — except here the weights are fixed instead of learned.

**Why we also change the embedding:** The bigram model used `Embedding(65, 65)` — each token mapped directly to 65 logits. Now we use a smaller `Embedding(65, 32)` to map tokens into a 32-dimensional space, do the attention there, then use a `Linear(32, 65)` layer to project back to 65 logits. This separation lets the model learn useful representations in the embedding space before making predictions.

**Why we add position embeddings:** Averaging is order-agnostic — the average of `[a, b, c]` is the same as `[c, a, b]`. We add a learned position embedding (`Embedding(block_size, 32)`) so the model knows *where* each token sits in the sequence.

This will actually perform *worse* than the bigram model (~2.7 vs ~2.5) — averaging blurs the signal. But it sets up the mechanic that becomes powerful once we make the weights learnable in Step 2.

In [ ]:
n_embd = 32

class AttentionLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        # Now embedding into a smaller 32-dim space, not directly into vocab_size
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        # Learned position embedding so the model knows token order
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        # Project from 32-dim embedding space back to 65 vocab logits
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx)            # (B, T, 32)
        pos_emb = self.position_embedding_table(torch.arange(T))  # (T, 32)
        x = tok_emb + pos_emb                                # (B, T, 32) — add position info

        # Build the uniform attention weight matrix
        tril = torch.tril(torch.ones(T, T))        # lower-triangular mask
        wei = torch.zeros((T, T))                   # start with equal weights
        wei = wei.masked_fill(tril == 0, float('-inf'))  # block future positions
        wei = F.softmax(wei, dim=-1)                # normalize rows to sum to 1
        x = wei @ x                                 # (T,T) @ (B,T,32) → (B,T,32)

        logits = self.lm_head(x)                     # (B, T, 65) — project to vocab

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B*T, C), targets.view(B*T))

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]  # crop to block_size (position embeddings are fixed-size)
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

attn_model = AttentionLanguageModel()
print(f'Parameters: {sum(p.numel() for p in attn_model.parameters()):,}')

### Training

Same training loop as Step 0. The only difference is the model — the training code doesn't change between steps.

In [ ]:
# Same loop as Step 0 — only the model changed, training logic stays identical
optimizer = torch.optim.AdamW(attn_model.parameters(), lr=1e-3)

for step in range(10000):
    xb, yb = get_batch('train')
    logits, loss = attn_model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if step % 2000 == 0 or step == 9999:
        print(f'Step {step:5d}  loss={loss.item():.4f}')

### Generate after training

Loss ~2.7 — *worse* than the bigram's ~2.5. Uniform averaging dilutes every token's contribution. By position 7, each token only contributes 1/8 of the signal. The model can't focus on what matters.

In [26]:
idx = torch.zeros((1, 1), dtype=torch.long)
print(decode(attn_model.generate(idx, max_new_tokens=300)[0].tolist()))


Hsowaasr tthtro-meeah nny  taoinr   ydowoohow u sgeuko.w iS

BchhoPueki mewtd os :?yni
T 
ostMygmrye ;Mpthcw hatitad m heirits c miyesa e  dnn iigwo rber idrr bgsfe:yl eouaarntnd I oohmm nda f mbcery,o,o  tlde h ai onbmuugshueec  rsinGOha  Lyoutrgrd, heoe tnofam k ainowdigo? 
nbn
pApudyio nai  kacth


## Step 2: Self-Attention

**What changes:** The weight matrix `wei` is no longer fixed — it's computed from the data, so the model learns which tokens to pay attention to.

**How it works:** In Step 1, every past token got equal weight — `"King:"` averaged `K`, `i`, `n`, `g`, `:` equally, even though the `:` is by far the most important for predicting what comes next (a speaker line). We need the model to learn that `:` matters more.

Self-attention does this with three projections per token:
- **Query** — "what am I looking for?" (computed by a learned linear layer)
- **Key** — "what do I contain?" (a different learned linear layer)
- **Value** — "what information should I pass along?" (a third learned linear layer)

The attention weight between token `i` and token `j` is the dot product `query[i] · key[j]`. Think of it like a search: the query is the search term, the key is the tag on each document. If they point in a similar direction (high dot product), that document is relevant and gets high weight. If they're unrelated (low dot product), it gets ignored.

After computing all dot products, we mask the future (same `tril` trick as before), apply softmax, and multiply by the value vectors. The result: each position gets a **learned, weighted** combination of past tokens instead of a uniform average.

**Why three separate projections?** Without the value projection, the model uses the same representation for "am I interesting?" (key) and "what should I contribute?" (value). Separating them lets the model say "this token is very relevant" (high key-query alignment) while contributing different information (through its value).

**Why `/ sqrt(head_size)`?** The dot products grow with the dimension of Q and K. With 16 dimensions, you're summing 16 products — the result can be large. Large values make softmax very "peaky" (one token gets nearly all the weight, others get ~0), which makes gradients tiny and training slow. Dividing by `sqrt(16) = 4` keeps the values in a range where softmax produces useful gradients.

**What the model learns:** During training, the Q, K, V weight matrices are updated by gradient descent just like any other weights. The model learns Q/K patterns like "queries from positions after a colon should match keys from capitalized-word positions" — but it discovers these patterns entirely on its own from the training data.

In [ ]:
head_size = 16

class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        # Three separate linear projections — no bias, as is conventional
        self.key   = nn.Linear(n_embd, head_size, bias=False)  # "what do I contain?"
        self.query = nn.Linear(n_embd, head_size, bias=False)  # "what am I looking for?"
        self.value = nn.Linear(n_embd, head_size, bias=False)  # "what should I contribute?"
        # The mask is not a learned parameter — register_buffer keeps it on the right device
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)    # (B, T, head_size) — every token broadcasts what it contains
        q = self.query(x)  # (B, T, head_size) — every token broadcasts what it wants

        # Dot product of each query with each key → (B, T, T) attention scores
        # High score = "this token is relevant to me"
        wei = q @ k.transpose(-2, -1) * head_size**-0.5  # scale to prevent sharp softmax
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))  # can't look at the future
        wei = F.softmax(wei, dim=-1)  # normalize: each row sums to 1

        # Weighted sum of value vectors — gather information from relevant tokens
        v = self.value(x)  # (B, T, head_size) — what each token actually contributes
        out = wei @ v      # (B, T, head_size) — the attended output
        return out


class SelfAttentionLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.sa_head = Head(head_size)
        # Note: lm_head maps from head_size (16), not n_embd (32)
        self.lm_head = nn.Linear(head_size, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T))
        x = tok_emb + pos_emb        # (B, T, 32)
        x = self.sa_head(x)           # (B, T, 16) — attention reduces the dimension
        logits = self.lm_head(x)      # (B, T, 65) — project to vocab

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B*T, C), targets.view(B*T))

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

sa_model = SelfAttentionLanguageModel()
print(f'Parameters: {sum(p.numel() for p in sa_model.parameters()):,}')

### Training

Same loop. The Q, K, V weight matrices are now learned parameters — backprop updates them just like any other weight. The model discovers which attention patterns reduce the loss.

In [ ]:
# Same loop — but now the model has learned Q, K, V projections
# Backprop will update those weights to discover useful attention patterns
optimizer = torch.optim.AdamW(sa_model.parameters(), lr=1e-3)

for step in range(10000):
    xb, yb = get_batch('train')
    logits, loss = sa_model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if step % 2000 == 0 or step == 9999:
        print(f'Step {step:5d}  loss={loss.item():.4f}')

### Generate after training

Loss ~2.3 — a clear improvement over both the bigram (~2.5) and uniform averaging (~2.7). The model can now focus on specific tokens. But one head = one attention pattern. It can't track multiple types of relationships at once.

In [34]:
idx = torch.zeros((1, 1), dtype=torch.long)
print(decode(sa_model.generate(idx, max_new_tokens=300)[0].tolist()))



adver ay own fowingoter gtoroulus,
Hesces.

Tout Es nsweciercon O.


ANICHERDURDIOULO:
GWar thidemy kecich towu coem tihee'd ounecrtencenen, ng.

SBor: ougel ayd waito hin ayoft!

EISS, Th;
Hen nd I, bem set hrken.
Gd argnt, my wem mungerer.

WAnd thorw we' i-cont ingo bed oche sthech weselitors so


## Step 3: Multi-Head Attention

**What changes:** Instead of one attention head, we run 4 in parallel and concatenate their outputs.

**How it works:** A single head has one set of Q, K, V weights, which produces one `(T, T)` attention pattern. That pattern might learn to look at the immediately preceding character, but then it *can't also* look at punctuation 5 positions back — it's one pattern per head.

Multi-head attention solves this by running several heads side by side. With `n_embd = 32` and 4 heads, each head works with `32 / 4 = 8` dimensions. Each head has its own Q, K, V weight matrices, so each one independently learns a different attention pattern.

**How do heads decide what to focus on?** They don't — nothing explicitly assigns roles to heads. Each head's Q, K, V weights are initialized randomly and updated by gradient descent independently. During training, if two heads happen to learn the same pattern, there's no loss improvement from the duplicate, so gradients push one of them toward a different pattern that actually helps. Over time, heads naturally specialize: one might learn to attend to the previous vowel, another to the nearest space, another to punctuation. This emergent specialization is a result of the optimization, not explicit programming.

After all heads run, their outputs (each 8-dimensional) are concatenated back to 32 dimensions, and a `proj` linear layer mixes them. This projection is important: without it, the heads can't combine their findings — head 1's output can't interact with head 2's. The projection lets the model learn things like "if head 1 found a colon AND head 3 found a capital letter, then..."

**Implementation:** `MultiHeadAttention` wraps an `nn.ModuleList` of `Head` instances. In `forward`, each head runs on the same input (they all see the same embeddings), the outputs are concatenated along the last dimension, and the `proj` layer mixes them. Since the concatenated output is `n_embd`-dimensional, `lm_head` maps from `n_embd` to `vocab_size` (unlike Step 2 where it mapped from `head_size`).

In [ ]:
n_heads = 4
head_size = n_embd // n_heads  # 32 // 4 = 8 dimensions per head

class MultiHeadAttention(nn.Module):
    def __init__(self, n_heads, head_size):
        super().__init__()
        # Each head has its own Q, K, V weights — they learn independently
        self.heads = nn.ModuleList([Head(head_size) for _ in range(n_heads)])
        # After concatenation, mix the heads' outputs so they can interact
        self.proj = nn.Linear(n_embd, n_embd)

    def forward(self, x):
        # Run all heads on the same input, concatenate their outputs
        # Each head returns (B, T, 8), concatenated → (B, T, 32)
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)  # (B, T, 32) — let heads combine their findings
        return out


class MultiHeadModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.sa_heads = MultiHeadAttention(n_heads, head_size)
        # Back to mapping from n_embd (32) since heads concatenate to full size
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T))
        x = tok_emb + pos_emb     # (B, T, 32)
        x = self.sa_heads(x)       # (B, T, 32) — multi-head attention
        logits = self.lm_head(x)   # (B, T, 65)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B*T, C), targets.view(B*T))
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

mh_model = MultiHeadModel()
print(f'Parameters: {sum(p.numel() for p in mh_model.parameters()):,}')

### Training

Same loop. Now 4 heads are trained in parallel — each head's Q, K, V weights are updated independently, and the projection layer learns to combine their outputs.

In [ ]:
# Same loop — now training 4 attention heads + their projection layer
# More parameters to train (8.6k vs 5k), but the loop itself is unchanged
optimizer = torch.optim.AdamW(mh_model.parameters(), lr=1e-3)

for step in range(10000):
    xb, yb = get_batch('train')
    logits, loss = mh_model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if step % 2000 == 0 or step == 9999:
        print(f'Step {step:5d}  loss={loss.item():.4f}')

### Generate after training

Loss ~2.2. Four heads tracking different patterns beats one head. The output starts showing longer word fragments.

In [37]:
idx = torch.zeros((1, 1), dtype=torch.long)
print(decode(mh_model.generate(idx, max_new_tokens=300)[0].tolist()))


Ane pron and to arle:
But with brand ised-o'm nould one,
My shenthould he ndll.

PORK:
Bord, thing the
Mis toighter wouted,
Go poothill.

That his
Theave fupn ma hath wichy to' sou re:vee mand, fron ase of an;
Whatoon I woull fourtecantt wleeon ore prow warbasmuciele itents ar rot be bothe hat hal h


## Step 4: Feed-Forward Layer

**What changes:** After attention, each token passes through a small neural network (two linear layers with a ReLU in between).

**How it works:** Attention computes a weighted sum of value vectors — this is a *linear* operation. It can gather information ("I see a colon 2 positions back, and a capital letter 5 positions back"), but it can't do nonlinear reasoning on what it found. The feed-forward network (FFN) adds that ability.

Think of it as two stages: attention answers "what should I look at?", the FFN answers "what do I do with what I found?". For example, the pattern "a colon after a capitalized word means a new speaker" requires combining multiple features in a nonlinear way — attention alone can't express that, but attention + FFN can.

The FFN processes each token position independently (no cross-token interaction — that's attention's job). It expands the dimension 4×, applies ReLU (which zeroes out negative values — this is the nonlinearity), then projects back down:

```
(B, T, 32) → Linear → (B, T, 128) → ReLU → (B, T, 128) → Linear → (B, T, 32)
```

The 4× expansion gives the network a wider "workspace" to compute intermediate features. This ratio comes from the original transformer paper and has become the standard.

**Implementation:** `FeedForward` is an `nn.Sequential` with three layers. In the model's `forward`, it's called right after attention: `x = self.sa_heads(x)` then `x = self.ffwd(x)`. The shapes stay the same — the FFN takes `(B, T, 32)` and returns `(B, T, 32)`.

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),  # expand: 32 → 128
            nn.ReLU(),                        # nonlinearity — zeroes out negatives
            nn.Linear(4 * n_embd, n_embd),   # project back: 128 → 32
        )

    def forward(self, x):
        return self.net(x)  # applied independently to each token position


class FFModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.sa_heads = MultiHeadAttention(n_heads, head_size)
        self.ffwd = FeedForward(n_embd)  # NEW: per-token processing after attention
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T))
        x = tok_emb + pos_emb     # (B, T, 32)
        x = self.sa_heads(x)       # (B, T, 32) — gather context from other tokens
        x = self.ffwd(x)           # (B, T, 32) — process what was gathered
        logits = self.lm_head(x)   # (B, T, 65)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B*T, C), targets.view(B*T))
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

ff_model = FFModel()
print(f'Parameters: {sum(p.numel() for p in ff_model.parameters()):,}')

### Training

Same loop. The FFN's weights are trained alongside the attention weights — backprop flows through FFN → attention → embeddings in one pass.

In [ ]:
# Same loop — the FFN adds ~8k new parameters (two linear layers)
# Backprop updates attention AND FFN weights together in each step
optimizer = torch.optim.AdamW(ff_model.parameters(), lr=1e-3)

for step in range(10000):
    xb, yb = get_batch('train')
    logits, loss = ff_model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if step % 2000 == 0 or step == 9999:
        print(f'Step {step:5d}  loss={loss.item():.4f}')

### Generate after training

Loss ~2.1. The FFN lets the model reason about gathered context — not just collect it. More recognizable words start appearing.

In [40]:
idx = torch.zeros((1, 1), dtype=torch.long)
print(decode(ff_model.generate(idx, max_new_tokens=300)[0].tolist()))


Bot buch it, hebut, it in toll thoom fall cor:
Pom Ient Wacill thice.

LO:
WANR:
Forn thelve:
Bled'd my-ast noce: mord, thea; thels, hrown mter you. LIZARBOLE:
Hire sow with the muing.

CASEN AULES:
Voe'ds ut that theood: abth ham in nots into shy steed you ther.

KING
She tay the gre huth in itin t


## Step 5: Residual Connections + Layer Norm

**What changes:** We wrap attention and feed-forward in residual connections and add layer normalization. This creates a `Block` — the repeating unit of a transformer.

**How it works:** Right now we have attention → feed-forward as a sequence: `x = ffwd(attention(x))`. This works fine with one layer, but when we want to stack many layers deep (Step 6), two problems appear:

**Problem 1: Vanishing gradients.** During training, PyTorch computes how each weight contributed to the error (via backpropagation). This gradient signal has to travel backwards through every layer. With many sequential transformations, the signal can shrink to near-zero — the deeper layers stop learning because the gradients are too small to move the weights. This is like a game of telephone where the message degrades with each hop.

**Fix: Residual connections.** Instead of `x = f(x)`, we do `x = x + f(x)`. The `+ x` creates a direct shortcut that gradients can flow through unchanged — even if `f(x)` has tiny gradients, the addition means the original signal still gets through. Think of it as a highway bypass around each layer. Each layer only needs to learn "what should I add to make this better", not "recompute everything from scratch".

**Problem 2: Unstable activations.** As values pass through many layers of linear transformations and ReLUs, they can drift to very large or very small ranges. This makes training unstable — the learning rate that works for one layer might be way too large or too small for another.

**Fix: Layer norm.** Before each sub-layer (attention, feed-forward), we normalize each token's values to zero mean and unit variance, then apply a learned scale and shift. This keeps the input to each layer in a predictable range regardless of what happened in previous layers. We normalize *before* the sub-layer (called "pre-norm"), which is the modern convention:

```python
x = x + attention(norm(x))   # normalize → attend → add back
x = x + feedforward(norm(x)) # normalize → FFN → add back
```

**The `Block` class** packages attention + FFN + residuals + norms into a single reusable unit. We also add a final `ln_f` layer norm after the block, before the output projection. In Step 6, we'll stack multiple blocks — that's where residuals and norms become essential.

In [ ]:
class Block(nn.Module):
    """One transformer block: attention + feed-forward, with residuals and norms."""
    def __init__(self, n_embd, n_heads):
        super().__init__()
        head_size = n_embd // n_heads
        self.sa = MultiHeadAttention(n_heads, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)  # normalize before attention
        self.ln2 = nn.LayerNorm(n_embd)  # normalize before feed-forward

    def forward(self, x):
        # Residual connection: add the input back after each sub-layer
        # Pre-norm: normalize before the sub-layer, not after
        x = x + self.sa(self.ln1(x))     # norm → attend → add back
        x = x + self.ffwd(self.ln2(x))   # norm → FFN → add back
        return x


class BlockModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.block = Block(n_embd, n_heads)    # one transformer block
        self.ln_f = nn.LayerNorm(n_embd)        # final norm before output projection
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T))
        x = tok_emb + pos_emb     # (B, T, 32)
        x = self.block(x)          # (B, T, 32) — attention + FFN with residuals
        x = self.ln_f(x)           # (B, T, 32) — final normalization
        logits = self.lm_head(x)   # (B, T, 65)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B*T, C), targets.view(B*T))
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

block_model = BlockModel()
print(f'Parameters: {sum(p.numel() for p in block_model.parameters()):,}')

### Training

Same loop. The residual connections mean gradients now have a direct path back to the embeddings — even through the attention and FFN layers. The LayerNorm parameters (a scale and shift per feature) are also learned.

In [ ]:
# Same loop — residuals give gradients a highway back to early layers
# LayerNorm adds a small number of parameters (scale + shift per feature)
optimizer = torch.optim.AdamW(block_model.parameters(), lr=1e-3)

for step in range(10000):
    xb, yb = get_batch('train')
    logits, loss = block_model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if step % 2000 == 0 or step == 9999:
        print(f'Step {step:5d}  loss={loss.item():.4f}')

### Generate after training

Loss ~1.9 — a big jump from the residual connections. Gradients flow better, so all the layers train more effectively. Real words and speaker-like patterns emerge.

In [43]:
idx = torch.zeros((1, 1), dtype=torch.long)
print(decode(block_model.generate(idx, max_new_tokens=300)[0].tolist()))


In devay herits ally magurss for the woul a the uprace!

Whord moman, your not, so live?

ENrecay negeediatimpletinss in as the firesteney proized non:
Come. I'll at, kine is Thate awar, pord it in thrhis my word! man notter the ner fuch your best is naugs me thour the crost sire coive pasten, sewel


## Step 6: Full GPT — Stacking Layers + Dropout + Scale Up

**What changes:** We stack 4 blocks deep, add dropout, and increase the model's dimensions. This is the complete GPT architecture.

**How stacking works:** A single block does one round of "gather context (attention) → process it (feed-forward)". With one block, the model can learn patterns like "character `h` often follows character `t`". By stacking 4 blocks, each block builds on the previous one's output, learning increasingly abstract patterns:
- Block 1 processes raw character embeddings — might learn simple patterns like common letter pairs
- Block 2 reads block 1's output — the input already encodes letter-pair info, so block 2 can recognize word fragments
- Block 3 and 4 can build on that — recognizing word-level patterns, speaker names, punctuation structure

This works because of the residual connections from Step 5: each block adds refinements to the signal rather than replacing it, so information from early layers is preserved.

**How dropout works:** During training, dropout randomly zeroes out 20% of values in a tensor. This sounds destructive, but it forces the model to not rely on any single feature — if a neuron might be disabled at any time, the model has to spread its knowledge across multiple neurons. This prevents overfitting (memorizing the training data instead of learning general patterns). During generation, dropout is automatically disabled (`model.eval()`) so the model uses all its capacity.

We add dropout in three places: after the attention weights (before multiplying by values), after the FFN output, and after the multi-head projection.

**Scaling up:** We increase the model's capacity to take advantage of the deeper architecture:
- `n_embd`: 32 → 64 — each token's representation is twice as rich
- `block_size`: 8 → 32 — the model sees 32 characters of context instead of 8
- `batch_size`: 32 → 64 — averaging gradients over more examples makes training more stable

**Implementation:** Since the hyperparameters change, we redefine all the component classes (GPTHead, GPTMultiHeadAttention, etc.) with the new dimensions and dropout. The model class stacks blocks with `nn.Sequential(*[GPTBlock(...) for _ in range(4)])`.

```
Tokens → Embedding + Position → Block × 4 → LayerNorm → Linear → Logits
```

In [ ]:
# Scaled-up hyperparameters
gpt_block_size = 32   # 4x more context than before
gpt_n_embd = 64       # 2x richer embeddings
gpt_n_heads = 4
gpt_n_layers = 4      # 4 blocks deep
gpt_head_size = gpt_n_embd // gpt_n_heads  # 16
dropout = 0.2          # randomly disable 20% of activations during training

class GPTHead(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.head_size = head_size
        self.key   = nn.Linear(gpt_n_embd, head_size, bias=False)
        self.query = nn.Linear(gpt_n_embd, head_size, bias=False)
        self.value = nn.Linear(gpt_n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(gpt_block_size, gpt_block_size)))
        self.dropout = nn.Dropout(dropout)  # NEW: dropout on attention weights

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * self.head_size**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)    # randomly zero some attention weights during training
        v = self.value(x)
        return wei @ v


class GPTMultiHeadAttention(nn.Module):
    def __init__(self, n_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([GPTHead(head_size) for _ in range(n_heads)])
        self.proj = nn.Linear(gpt_n_embd, gpt_n_embd)
        self.dropout = nn.Dropout(dropout)  # NEW: dropout after projection

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out


class GPTFeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),  # NEW: dropout after FFN
        )

    def forward(self, x):
        return self.net(x)


class GPTBlock(nn.Module):
    def __init__(self, n_embd, n_heads):
        super().__init__()
        head_size = n_embd // n_heads
        self.sa = GPTMultiHeadAttention(n_heads, head_size)
        self.ffwd = GPTFeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


class GPTLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, gpt_n_embd)
        self.position_embedding_table = nn.Embedding(gpt_block_size, gpt_n_embd)
        # Stack 4 blocks — this is where depth comes from
        self.blocks = nn.Sequential(*[GPTBlock(gpt_n_embd, gpt_n_heads) for _ in range(gpt_n_layers)])
        self.ln_f = nn.LayerNorm(gpt_n_embd)
        self.lm_head = nn.Linear(gpt_n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T))
        x = tok_emb + pos_emb   # (B, T, 64)
        x = self.blocks(x)       # (B, T, 64) — pass through all 4 blocks
        x = self.ln_f(x)         # (B, T, 64) — final normalization
        logits = self.lm_head(x) # (B, T, 65)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B*T, C), targets.view(B*T))
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -gpt_block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

gpt_model = GPTLanguageModel()
print(f'Parameters: {sum(p.numel() for p in gpt_model.parameters()):,}')

### Training

First change to the training code: we need a new `get_batch` that uses the larger `block_size = 32` and `batch_size = 64`. The model is also ~12× bigger (210k params), so each step takes longer — but the loop itself is the same. Dropout is active during training and automatically disabled during generation.

In [ ]:
# New get_batch for the scaled-up hyperparameters
gpt_batch_size = 64  # 2× more sequences per batch for more stable gradients

def get_batch_gpt(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - gpt_block_size, (gpt_batch_size,))
    x = torch.stack([d[i:i+gpt_block_size] for i in ix])    # 32 chars of context (was 8)
    y = torch.stack([d[i+1:i+gpt_block_size+1] for i in ix])
    return x, y

# Same training loop — 210k parameters across 4 blocks, all updated each step
# Dropout randomly disables neurons during training (prevents overfitting)
optimizer = torch.optim.AdamW(gpt_model.parameters(), lr=1e-3)

for step in range(10000):
    xb, yb = get_batch_gpt('train')
    logits, loss = gpt_model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if step % 2000 == 0 or step == 9999:
        print(f'Step {step:5d}  loss={loss.item():.4f}')

### Generate after training

Loss ~1.6 — the best yet. With 4 stacked blocks, more context (32 chars), and 50× more parameters, the model produces recognizable English words, speaker names with colons, and Shakespeare-like sentence structure.

In [46]:
idx = torch.zeros((1, 1), dtype=torch.long)
print(decode(gpt_model.generate(idx, max_new_tokens=500)[0].tolist()))



SAMILLO:
Whomet say.

LADHAGUD ET woull I be sole, abet.

RACHIArwiLLIFF:
I not it lord
Thy will know; 
The driviant of my duke of the prements advy heart in froef's grot he foul I passure the son. My are these statdly, Clorame tear good.

MARGLUCIOA:
No w is way softer stands. That go is defet haour;
the worst: here oft, with when heart.

SICINIUS:
I has can with head.

AUFIO:
Thou shalce your love, heir Haplemay you the friend
For Servant unling my bellow; who is a teir, lessir for be they ri


## Summary

| Step | What we added | Parameters | Loss |
|------|--------------|------------|------|
| 0 | Bigram — no context | 4,225 | ~2.5 |
| 1 | Uniform averaging | 4,481 | ~2.7 |
| 2 | Learned attention (Q, K, V) | 4,977 | ~2.3 |
| 3 | Multiple attention heads | 8,609 | ~2.2 |
| 4 | Feed-forward layer | 16,961 | ~2.1 |
| 5 | Residual connections + layer norm | 17,153 | ~1.9 |
| 6 | **4 layers + dropout + scale up** | **209,729** | **~1.6** |

Step 1 is intentionally *worse* than the baseline — uniform averaging blurs everything. The real breakthrough is Step 2, when the model learns which tokens matter.

This is the same architecture (at much larger scale) behind GPT-2, GPT-3, and similar models.

## Next steps

Three directions to take this further:

1. **Train longer on GPU** — our GPT model barely scratches the surface at 10k steps on CPU. Move to GPU with `model.to('cuda')`, train for 100k+ steps, and you'll see noticeably more coherent Shakespeare with real words and sentence structure.

2. **Subword tokenizer (BPE)** — character-level tokenization means the model has to learn to spell every word from individual letters. A subword tokenizer like [tiktoken](https://github.com/openai/tiktoken) groups common character sequences into single tokens (e.g., `"the"` → one token instead of three), letting the model learn much faster with a ~50k vocabulary.

3. **Train on a real task** — instead of open-ended text generation, train on something with a measurable outcome: simple arithmetic (`"2+3=" → "5"`), translation between toy languages, or text completion with a structured prompt format. This makes it easier to evaluate whether the model actually learned the pattern vs. just memorizing statistics.